> 在线讲义：[第 23 章 · 调优方法论：先算账，再测量，只改瓶颈](https://wikieden.github.io/pytorch-inside-out/lessons/23-perf-method.html) · 本书《PyTorch 从里到外》

# 第 23 章 · 调优方法论：先算账，再测量，只改瓶颈

本章回答：训练"快不快"用什么数衡量？MFU 怎么算，多少算好？拿到一个慢的训练脚本，按什么顺序找问题？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/23-perf-method.md`（讲解、动画、练习都在讲义里）。对应原文：Ultra-Scale Playbook §First Steps（MFU / HFU 定义在 `ultrascale-playbook.md` 第 420–440 行）；多机报告 §1.2 算力账（`distributed-pretraining/notes/03-engineering-report.md` 第 68–74 行，含 DeepSeek-V3 MFU 参考值）；AISPE ch13 引言（`ch13.md` 第 1–18 行）；Kaplan et al. 2020「Scaling Laws」§2.1（6N 的出处，〔在线〕arXiv:2001.08361）

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 6N | 前向 2N、反向 4N | `train_flops_per_token(7e9)` | 1.4e10 / 2.8e10 / 4.2e10 |
| 第二步 MFU | 6N·tok/s ÷ (G·peak) | `mfu(7e9, 96_000, 8, 989e12)` | 51.0% |
| 附录 训练时间 | 6ND ÷ (G·peak·MFU) | `train_time_days(...)` | 约 120 天 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `mfu`**（来自 `ch23_perf_method.py`）：下面要用到的小工具。

In [ ]:
def mfu(n_params: float, tokens_per_s: float, n_gpus: int, peak_flops: float) -> float:
    return 6 * n_params * tokens_per_s / (n_gpus * peak_flops)

**函数 `train_flops_per_token`**（来自 `ch23_perf_method.py`）：下面要用到的小工具。

In [ ]:
def train_flops_per_token(n_params: float) -> dict:
    return dict(forward=2 * n_params, backward=4 * n_params, total=6 * n_params)

**函数 `train_time_days`**（来自 `ch23_perf_method.py`）：下面要用到的小工具。

In [ ]:
def train_time_days(n_params, n_tokens, n_gpus, peak_flops, mfu_value) -> float:
    return 6 * n_params * n_tokens / (n_gpus * peak_flops * mfu_value) / 86400

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 7B 每 token：", train_flops_per_token(7e9), "；训 1 万亿 token 共", f"{6 * 7e9 * 1e12:.1e}", "FLOPs")

In [ ]:
print("\n第二步 7B、8×H100、9.6 万 token/s → MFU", f"{mfu(7e9, 96_000, 8, 989e12):.1%}")
print("同样配置训 1 万亿 token 约", round(train_time_days(7e9, 1e12, 8, 989e12, 0.51), 1), "天")

## 3. 代码里才看得到的

1. **峰值要用对精度的数**：H100 bf16 稠密 989 TFLOPS；用稀疏的 1979 会让 MFU 少一半。

## 4. 自己改一改

- 算 70B、1024 张 H100、MFU 40% 训 15 万亿 token 要几天。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 每个 token 六倍参数：前向二、反向四；
> MFU 等于六 N 乘吞吐，除以卡数乘峰值；dense 五成好，MoE 四成一流；
> 闲着、访存、算、通信，先看时间线；改一处测一次，别忘对 loss。